In [0]:
dbutils.library.restartPython()

# Spike — validação do padrão de import entre simuladores

Confirma que a conversão de Notebook (`%run`) para arquivo puro (`import`)
funciona corretamente dentro do Databricks Repo, sem poluir a saída com
markdown de outros arquivos. Testa `SimuladorERP.gerar_seed()` isoladamente,
sem gravação real (Volume da Landing Zone ainda não existe).

Ver ADR-003 (Programação Orientada a Objetos).

In [0]:
from src.simuladores.simulador_erp import SimuladorERP

simulador = SimuladorERP(spark=spark, dbutils=dbutils)
resultado = simulador.gerar_seed()

print(resultado)


In [0]:
resultado_seed = simulador.executar_seed()
print(resultado_seed)

In [0]:
from datetime import date
from src.simuladores.simulador_tms import SimuladorTMS

simulador_tms = SimuladorTMS(spark=spark, dbutils=dbutils)
resultado = simulador_tms.gerar_dia(date(2026, 8, 3))
print(resultado)

In [0]:
resultado = simulador_tms.gerar_dia(date(2026, 8, 3))
print(resultado)

In [0]:
caminho_erp = simulador.caminho_landing(date(2026, 8, 3))
caminho_tms = simulador_tms.caminho_landing(date(2026, 8, 3))

df_remessas = spark.read.json(f"{caminho_tms}/tms_remessas.json")
df_notas = spark.read.json(f"{caminho_erp}/erp_notas_expedicao.json")
df_lotes = spark.read.json(f"{caminho_erp}/erp_lotes_producao.json")

# mapa veículo -> refrigerado (do seed, direto em Python)
veiculos_refrigerados = {"VEI-003", "VEI-005"}

# junta remessa -> nota -> lote -> produto, e marca se o produto exige cadeia fria
df_join = (
    df_remessas.join(df_notas, "nota_expedicao_id")
    .join(df_lotes.select("lote_id", "produto_id"), "lote_id", "left")
)

df_pandas = df_join.select("remessa_id", "veiculo_id", "produto_id").toPandas()
df_pandas["exige_cadeia_fria"] = df_pandas["produto_id"] == "PROD-002"  # Imunorax
df_pandas["veiculo_ok"] = df_pandas["veiculo_id"].isin(veiculos_refrigerados)

violacoes = df_pandas[df_pandas["exige_cadeia_fria"] & ~df_pandas["veiculo_ok"]]
print(f"Total de remessas de produto com cadeia fria: {df_pandas['exige_cadeia_fria'].sum()}")
print(f"Violações (veículo errado): {len(violacoes)}")